Dragos Nicolier & Alan-Ilian-Imran Traore

Ce notebook est dédié à la création, l'entraînement et l'évaluation du modèle de Machine Learning capable de prédire les retards de trains de la SNCF.

Objectif : Prédire le retard à l'arrivée (en minutes) en fonction des caractéristiques du trajet.

# Documentation du Modèle : Entraînement de l'Intelligence Artificielle (Machine Learning)

## 1. Qu'est-ce que ce modèle ?
Ce notebook représente le cœur prédictif du projet TARDIS. Il s'agit du pipeline d'apprentissage automatique (Machine Learning) visant à entraîner des algorithmes mathématiques pour qu'ils découvrent les schémas de retards historiques, et soient capables de prédire un futur retard (en minutes) sur des données inconnues.

## 2. Qui le développe et à qui sert-il ?
- **Développé par :** L'équipe de Data Scientists (Conception des algorithmes et évaluation des métriques).
- **À qui ça sert :** Il sert de "cerveau" à l'application web finale (Dashboard Streamlit). In fine, il sert le grand public (pour anticiper son trajet) et la direction SNCF (pour simuler des scénarios d'impact).

## 3. Sur quelles données se base-t-il ?
Il se base exclusivement sur le fichier `cleaned_dataset.csv` qui a été préalablement purgé de ses erreurs, valeurs manquantes et incohérences dans le pipeline précédent (l'EDA).

## 4. Pourquoi ce modèle et pas un autre (Axe de réflexion et Optimisation) ?
Nous mettons en compétition deux approches :
- **La Régression Linéaire :** Utilisée comme modèle de base (*baseline*). Elle est simple, extrêmement rapide, mais suppose que les causes de retard s'additionnent de manière linéaire, ce qui est souvent faux dans la réalité.
- **La Forêt Aléatoire (Random Forest Regressor) :** C'est le modèle optimisé que nous privilégions. Pourquoi ? Parce que c'est un modèle "ensembliste" (composé de multiples arbres de décision) capable de capturer des relations non-linéaires complexes (ex: la combinaison spécifique d'un jour de week-end avec un problème d'infrastructure). Il est extrêmement robuste au bruit, évite le surapprentissage, et reste suffisamment léger pour fournir des prédictions instantanées en production.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

print("Bibliothèques importées avec succès ! Prêt pour la suite.")

Bibliothèques importées avec succès ! Prêt pour la suite.


## Ingestion des Données Propres

**Comment ça fonctionne (Technique) :**
Nous chargeons le fichier CSV nettoyé en mémoire.

In [2]:
df = pd.read_csv('cleaned_dataset.csv')

print("Aperçu de nos VRAIES données :")
display(df.head())

print("\nListe des colonnes disponibles :")
print(df.columns)

Aperçu de nos VRAIES données :


,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,...,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",year,month,weekday,is_weekend
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,...,36.134454,31.092437,10.924370,15.966387,NaN,0.840336,2018.0,1.0,0.0,0
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,...,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333,2018.0,1.0,0.0,0
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,...,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018.0,1.0,0.0,0
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,...,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852,2018.0,1.0,0.0,0
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,...,15.789474,45.614035,NaN,15.789474,1.754386,1.754386,2018.0,1.0,0.0,0



Liste des colonnes disponibles :
Index(['Date', 'Service', 'Departure station', 'Arrival station',
       'Average journey time', 'Number of scheduled trains',
       'Number of cancelled trains', 'Number of trains delayed at departure',
       'Average delay of late trains at departure',
       'Average delay of all trains at departure',
       'Number of trains delayed at arrival',
       'Average delay of late trains at arrival',
       'Average delay of all trains at arrival',
       'Number of trains delayed > 15min',
       'Average delay of trains > 15min (if competing with flights)',
       'Number of trains delayed > 30min', 'Number of trains delayed > 60min',
       'Pct delay due to external causes', 'Pct delay due to infrastructure',
       'Pct delay due to traffic management', 'Pct delay due to rolling stock',
       'Pct delay due to station management and equipment reuse',
       'Pct delay due to passenger handling (crowding, disabled persons, connections)',
       'y

## Préparation des Matrices et Coupe Train/Test (Data Splitting)

**Qu'est-ce que c'est :**
La phase de transformation finale où l'on sépare ce que l'on veut deviner (la cible "y") de ce que l'on sait (les caractéristiques "X"), puis où l'on coupe les données en deux lots : Apprentissage (80%) et Test (20%).

**Pourquoi c'est comme ça :**
1. **Filtre numérique (`select_dtypes`) et `fillna(0)`** : L'algorithme mathématique ne sait lire que des nombres stricts. Tout "trou" restant est considéré comme une absence de problème (donc 0). Les textes non encodés sont ignorés pour éviter un crash matriciel.
2. **Coupe Train/Test** : Si on évalue un modèle sur les données qu'il a déjà vues, il triche (surapprentissage ou *overfitting*). On doit lui cacher 20% des trajets pour vérifier s'il est capable de "généraliser" son intelligence sur des cas qu'il ne connaît pas.

In [3]:
nom_colonne_cible = "Average delay of all trains at arrival"
df = df.fillna(0)
df_numeric = df.select_dtypes(include=[np.number])
X = df_numeric.drop(nom_colonne_cible, axis=1) 
y = df_numeric[nom_colonne_cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Nombre de trajets pour l'entraînement du modèle : {X_train.shape[0]}")
print(f"Nombre de trajets pour tester le modèle : {X_test.shape[0]}")

Nombre de trajets pour l'entraînement du modèle : 9440
Nombre de trajets pour tester le modèle : 2360


## Entraînement et Évaluation des Modèles (Model Fitting & Scoring)

**Comment ça fonctionne :**
La méthode ".fit()" lance l'optimisation mathématique : l'algorithme ajuste ses poids internes pour minimiser l'erreur entre sa prédiction et la vraie valeur de retard. Ensuite, ".predict()" génère des estimations sur nos 20% de données cachées.

**Quand ça peut être utilisé (Métriques d'évaluation) :**
Nous utilisons des métriques "métier" claires :
- **Le MAE (Erreur Absolue Moyenne) :** Indique concrètement à l'utilisateur de combien de minutes le modèle se trompe en moyenne. C'est l'indicateur le plus parlant commercialement.
- **Le RMSE (Erreur Quadratique Moyenne) :** Pénalise très fortement les grosses erreurs (les prédictions aberrantes).
- **Le R² (Coefficient de Détermination) :** Indique le pourcentage de la variance expliquée. Plus il se rapproche de 1, plus le modèle "comprend" les véritables causes des retards.

In [4]:
modele_lineaire = LinearRegression()
modele_foret = RandomForestRegressor(random_state=42)

print("Entraînement des modèles en cours...\n")
modele_lineaire.fit(X_train, y_train)
modele_foret.fit(X_train, y_train)

predictions_lineaire = modele_lineaire.predict(X_test)
predictions_foret = modele_foret.predict(X_test)

print("--- RÉSULTATS : RÉGRESSION LINÉAIRE ---")
print(f"MAE (Erreur Moyenne) : {mean_absolute_error(y_test, predictions_lineaire):.2f} minutes")
print(f"RMSE (Erreur Quadratique) : {np.sqrt(mean_squared_error(y_test, predictions_lineaire)):.2f} minutes")
print(f"R² (Précision globale) : {r2_score(y_test, predictions_lineaire):.2f}")

print("\n--- RÉSULTATS : FORÊT ALÉATOIRE (Random Forest) ---")
print(f"MAE (Erreur Moyenne) : {mean_absolute_error(y_test, predictions_foret):.2f} minutes")
print(f"RMSE (Erreur Quadratique) : {np.sqrt(mean_squared_error(y_test, predictions_foret)):.2f} minutes")
print(f"R² (Précision globale) : {r2_score(y_test, predictions_foret):.2f}")

Entraînement des modèles en cours...

--- RÉSULTATS : RÉGRESSION LINÉAIRE ---
MAE (Erreur Moyenne) : 2.01 minutes
RMSE (Erreur Quadratique) : 10.22 minutes
R² (Précision globale) : 0.08

--- RÉSULTATS : FORÊT ALÉATOIRE (Random Forest) ---
MAE (Erreur Moyenne) : 1.50 minutes
RMSE (Erreur Quadratique) : 9.98 minutes
R² (Précision globale) : 0.13


## Sérialisation et Déploiement (Model Export)

**Qu'est-ce que c'est :**
La "congélation" (sérialisation) du modèle d'Intelligence Artificielle entraîné (le Random Forest) dans un fichier physique ".pkl" via la librairie "joblib".

**Où c'est utilisé et Pourquoi c'est comme ça :**
Ce fichier `.pkl` va être directement importé par le Dashboard Streamlit (l'interface utilisateur). 
*Pourquoi ?* Pour des raisons d'optimisation (coûts et temps de calcul). Il serait impensable de devoir re-charger le dataset et ré-entraîner toute la forêt d'arbres décisionnels à chaque fois qu'un utilisateur clique sur le bouton "Prédire mon retard" sur le site web. Le modèle sauvegardé permet une prédiction instantanée en production (Inférence).

In [5]:
fichier_modele = 'model.pkl'
joblib.dump(modele_foret, fichier_modele)

print(f"Félicitations ! Le modèle a été sauvegardé avec succès sous le nom : {fichier_modele}")

Félicitations ! Le modèle a été sauvegardé avec succès sous le nom : model.pkl
